In [ ]:
!pip install contractions


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.1/345.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.2 MB/s eta 0:00:00


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import subprocess
import sys
import os

def install_packages():
    packages = [
        'nltk==3.8.1',
        'wordcloud==1.9.2',
        'imbalanced-learn==0.11.0',
        'contractions==0.1.73',
        'fasttext==0.9.2',
        'gensim==4.3.2',
        'scikit-learn==1.3.2',
        'xgboost==2.0.3',
        'matplotlib==3.7.2',
        'seaborn==0.12.2',
        'pandas==2.0.3',
        'numpy==1.24.3',
        'textblob==0.17.1'
    ]

    print("Installing packages...")
    for package in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', package],
                                  stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print(f"✓ {package.split('==')[0]}")
        except:
            print(f"⚠ Failed: {package}")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import re
from collections import Counter
import pickle
from datetime import datetime

import nltk
import contractions
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize, sent_tokenize

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier, StackingClassifier, RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_curve, auc)
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from scipy.sparse import hstack, csr_matrix

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from textblob import TextBlob
    TEXTBLOB_AVAILABLE = True
except ImportError:
    TEXTBLOB_AVAILABLE = False

try:
    import fasttext
    import gensim.downloader as api
    FASTTEXT_AVAILABLE = True
except ImportError:
    FASTTEXT_AVAILABLE = False

nltk_downloads = ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab', 'averaged_perceptron_tagger']
for item in nltk_downloads:
    try:
        nltk.download(item, quiet=True)
    except:
        pass

print("Libraries loaded successfully!")

OUTPUT_DIR = 'research_outputs'
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Created output directory: {OUTPUT_DIR}")

class Config:
    RANDOM_STATE = 42
    TEST_SIZE = 0.15  # Reduced to have more training data
    VALIDATION_SIZE = 0.15

    # OPTIMIZED TF-IDF parameters
    TFIDF_MAX_FEATURES = 10000  # Increased for better representation
    TFIDF_NGRAM_RANGE = (1, 4)  # Extended to 4-grams
    TFIDF_MIN_DF = 1  # More inclusive
    TFIDF_MAX_DF = 0.90  # Less aggressive filtering

    # OPTIMIZED HHO parameters
    N_HAWKS = 30  # Increased for better exploration
    MAX_ITERATIONS = 50  # More iterations for convergence
    FEATURE_THRESHOLD = 0.50  # Lower threshold for more features

    # OPTIMIZED ETC parameters
    ETC_N_ESTIMATORS = 500  # Significantly increased
    ETC_MAX_DEPTH = 35  # Deeper trees
    ETC_MIN_SAMPLES_SPLIT = 3
    ETC_MIN_SAMPLES_LEAF = 1
    ETC_MAX_FEATURES = 'sqrt'

    # OPTIMIZED XGBoost parameters
    XGB_N_ESTIMATORS = 500
    XGB_MAX_DEPTH = 10
    XGB_LEARNING_RATE = 0.03
    XGB_SUBSAMPLE = 0.85
    XGB_COLSAMPLE_BYTREE = 0.85
    XGB_MIN_CHILD_WEIGHT = 1
    XGB_GAMMA = 0.1

    # OPTIMIZED SVM parameters
    SVM_C = 15.0
    SVM_KERNEL = 'rbf'
    SVM_GAMMA = 'scale'

    # OPTIMIZED SMOTE parameters
    SMOTE_K_NEIGHBORS = 7
    USE_SMOTE_ENN = True  # Use SMOTEENN for better results

def load_dataset():
    try:
        df = pd.read_csv('reviews.csv')
        print("Dataset loaded from reviews.csv")
        print(f"Shape: {df.shape}")

        required_columns = ['text_', 'label']
        if not all(col in df.columns for col in required_columns):
            raise ValueError("Missing required columns")

        return df

    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Please make sure 'reviews.csv' is uploaded")
        return None

class AdvancedTextPreprocessor:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        try:
            self.stop_words = set(stopwords.words('english'))
            # Keep negations and intensifiers
            self.stop_words -= {'not', 'no', 'never', 'neither', 'very', 'most', 'least',
                                'always', 'really', 'too', 'quite', 'just', 'only'}
        except:
            self.stop_words = set([
                'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
                'for', 'of', 'with', 'by', 'this', 'that', 'these', 'those'
            ])

    def expand_contractions(self, text):
        try:
            return contractions.fix(text)
        except:
            replacements = {
                "don't": "do not", "won't": "will not", "can't": "cannot",
                "n't": " not", "'re": " are", "'ve": " have", "'ll": " will",
                "'d": " would", "'m": " am", "it's": "it is", "that's": "that is"
            }
            for contraction, expansion in replacements.items():
                text = text.replace(contraction, expansion)
            return text

    def clean_text(self, text):
        if pd.isna(text) or text == '':
            return ''

        text = str(text).lower()

        # Remove URLs
        text = re.sub(r'http\S+|www\S+|https\S+', '', text)

        # Remove emails
        text = re.sub(r'\S+@\S+', '', text)

        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)

        # Handle repeated characters (keep some for emphasis)
        text = re.sub(r'(.)\1{4,}', r'\1\1\1', text)

        # Keep important punctuation and symbols
        text = re.sub(r'[^\w\s\.\!\?\,\'\-\$]', ' ', text)

        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text.strip())

        return text

    def tokenize_and_process(self, text):
        try:
            tokens = word_tokenize(text)
        except:
            tokens = text.split()

        processed = []
        for token in tokens:
            if len(token) < 2 or (token.isdigit() and len(token) < 4):
                continue

            if token not in self.stop_words or token in {'not', 'no', 'never', 'very', 'really'}:
                try:
                    token = self.lemmatizer.lemmatize(token, pos='v')
                    token = self.lemmatizer.lemmatize(token, pos='n')
                except:
                    pass
                processed.append(token)

        return processed

    def preprocess(self, text):
        text = self.expand_contractions(text)
        text = self.clean_text(text)
        tokens = self.tokenize_and_process(text)
        return ' '.join(tokens)

class EnhancedFeatureExtractor:
    def __init__(self, config=Config()):
        self.config = config

        # TF-IDF Vectorizer with optimized parameters
        self.tfidf_vectorizer = TfidfVectorizer(
            max_features=config.TFIDF_MAX_FEATURES,
            ngram_range=config.TFIDF_NGRAM_RANGE,
            min_df=config.TFIDF_MIN_DF,
            max_df=config.TFIDF_MAX_DF,
            sublinear_tf=True,
            use_idf=True,
            norm='l2'
        )

        # Character n-grams
        self.char_vectorizer = TfidfVectorizer(
            analyzer='char',
            ngram_range=(3, 6),
            max_features=1500,
            min_df=2
        )

        # Word-level count vectorizer for additional features
        self.count_vectorizer = CountVectorizer(
            max_features=2000,
            ngram_range=(1, 2),
            min_df=2
        )

        self.use_fasttext = FASTTEXT_AVAILABLE
        self.fasttext_model = None

        if self.use_fasttext:
            try:
                print("Loading FastText model...")
                self.fasttext_model = api.load("fasttext-wiki-news-subwords-300")
                print("FastText model loaded successfully")
            except Exception as e:
                print(f"Could not load FastText: {e}")
                self.use_fasttext = False

    def extract_advanced_linguistic_features(self, texts):
        features = []

        for text in texts:
            feature_vector = []

            words = text.split()
            sentences = sent_tokenize(text) if text else []

            # Basic statistics (normalized)
            text_len = len(text)
            word_count = len(words)
            sent_count = len(sentences)

            feature_vector.append(text_len / 1000.0)  # Normalized
            feature_vector.append(word_count / 100.0)
            feature_vector.append(sent_count / 10.0)
            feature_vector.append(np.mean([len(word) for word in words]) if words else 0)
            feature_vector.append(np.mean([len(sent.split()) for sent in sentences]) if sentences else 0)

            # Punctuation features (normalized)
            feature_vector.append(text.count('!') / max(text_len, 1) * 100)
            feature_vector.append(text.count('?') / max(text_len, 1) * 100)
            feature_vector.append(text.count('.') / max(text_len, 1) * 100)
            feature_vector.append(text.count(',') / max(text_len, 1) * 100)
            feature_vector.append(text.count('...') / max(word_count, 1) * 100)
            feature_vector.append(text.count(';') / max(text_len, 1) * 100)
            feature_vector.append(text.count(':') / max(text_len, 1) * 100)

            # Lexical diversity
            unique_words = len(set(words)) if words else 0
            feature_vector.append(unique_words / max(word_count, 1))
            feature_vector.append(word_count - unique_words if words else 0)

            # Extended sentiment words
            positive_words = ['great', 'excellent', 'amazing', 'perfect', 'love', 'best',
                              'wonderful', 'awesome', 'fantastic', 'outstanding', 'superb',
                              'brilliant', 'incredible', 'magnificent', 'exceptional']
            negative_words = ['bad', 'terrible', 'awful', 'worst', 'hate', 'poor',
                              'horrible', 'disappointing', 'useless', 'waste', 'never']

            pos_count = sum(text.count(word) for word in positive_words)
            neg_count = sum(text.count(word) for word in negative_words)

            feature_vector.append(pos_count / max(word_count, 1) * 100)
            feature_vector.append(neg_count / max(word_count, 1) * 100)
            feature_vector.append(pos_count - neg_count)

            # Case features
            uppercase_chars = sum(1 for c in text if c.isupper())
            feature_vector.append(uppercase_chars / max(text_len, 1))

            # Digit features
            digits = sum(1 for c in text if c.isdigit())
            feature_vector.append(digits / max(text_len, 1))

            # Repetition features
            repeated_words = len([w for w in words if words.count(w) > 1])
            feature_vector.append(repeated_words / max(word_count, 1))

            # Multiple exclamations/questions (strong fake indicator)
            feature_vector.append(len(re.findall(r'!{2,}', text)))
            feature_vector.append(len(re.findall(r'\?{2,}', text)))
            feature_vector.append(len(re.findall(r'\.{3,}', text)))

            # Word length statistics
            word_lengths = [len(w) for w in words] if words else [0]
            feature_vector.append(np.std(word_lengths))
            feature_vector.append(np.max(word_lengths) if word_lengths else 0)
            feature_vector.append(np.min(word_lengths) if word_lengths else 0)

            # Sentiment analysis
            if TEXTBLOB_AVAILABLE:
                try:
                    blob = TextBlob(text)
                    feature_vector.append(blob.sentiment.polarity)
                    feature_vector.append(blob.sentiment.subjectivity)
                    feature_vector.append(abs(blob.sentiment.polarity))  # Absolute polarity
                except:
                    feature_vector.extend([0, 0, 0])
            else:
                feature_vector.extend([0, 0, 0])

            # Average word frequency
            word_freq = Counter(words)
            avg_freq = np.mean(list(word_freq.values())) if word_freq else 0
            max_freq = np.max(list(word_freq.values())) if word_freq else 0
            feature_vector.append(avg_freq)
            feature_vector.append(max_freq)

            # Sentence length statistics
            sent_lengths = [len(sent.split()) for sent in sentences] if sentences else [0]
            feature_vector.append(np.std(sent_lengths))
            feature_vector.append(np.max(sent_lengths) if sent_lengths else 0)

            # Capital letter patterns
            capital_words = sum(1 for w in words if w.isupper() and len(w) > 1)
            feature_vector.append(capital_words / max(word_count, 1))

            # Special character density
            special_chars = len(re.findall(r'[^a-zA-Z0-9\s]', text))
            feature_vector.append(special_chars / max(text_len, 1))

            # Emphasis patterns (fake reviews often overuse)
            feature_vector.append(text.count('!!!'))
            feature_vector.append(text.count('amazing'))
            feature_vector.append(text.count('perfect'))
            feature_vector.append(text.count('recommend'))

            features.append(feature_vector)

        return np.array(features)

    def get_fasttext_embeddings(self, texts):
        if not self.use_fasttext or self.fasttext_model is None:
            return np.zeros((len(texts), 300))

        embeddings = []
        for text in texts:
            words = text.split()
            if not words:
                embeddings.append(np.zeros(300))
                continue

            word_vectors = []
            for word in words:
                try:
                    word_vectors.append(self.fasttext_model[word])
                except:
                    pass

            if word_vectors:
                # Use both mean and max pooling
                mean_vec = np.mean(word_vectors, axis=0)
                embeddings.append(mean_vec)
            else:
                embeddings.append(np.zeros(300))

        return np.array(embeddings)

    def fit_transform(self, texts):
        print("Extracting TF-IDF features...")
        tfidf_features = self.tfidf_vectorizer.fit_transform(texts)

        print("Extracting character n-gram features...")
        char_features = self.char_vectorizer.fit_transform(texts)

        print("Extracting count features...")
        count_features = self.count_vectorizer.fit_transform(texts)

        print("Extracting advanced linguistic features...")
        linguistic_features = self.extract_advanced_linguistic_features(texts)

        features_list = [
            tfidf_features,
            char_features,
            count_features,
            csr_matrix(linguistic_features)
        ]

        if self.use_fasttext:
            print("Computing FastText embeddings...")
            embeddings = self.get_fasttext_embeddings(texts)
            features_list.append(csr_matrix(embeddings))

        print("Combining all features...")
        combined_features = hstack(features_list)

        return combined_features

    def transform(self, texts):
        tfidf_features = self.tfidf_vectorizer.transform(texts)
        char_features = self.char_vectorizer.transform(texts)
        count_features = self.count_vectorizer.transform(texts)
        linguistic_features = self.extract_advanced_linguistic_features(texts)

        features_list = [
            tfidf_features,
            char_features,
            count_features,
            csr_matrix(linguistic_features)
        ]

        if self.use_fasttext:
            embeddings = self.get_fasttext_embeddings(texts)
            features_list.append(csr_matrix(embeddings))

        combined_features = hstack(features_list)
        return combined_features

class HarrisHawksOptimizer:
    def __init__(self, n_hawks=30, max_iterations=50, random_state=42):
        self.n_hawks = n_hawks
        self.max_iterations = max_iterations
        self.random_state = random_state
        np.random.seed(random_state)

        self.best_position = None
        self.best_fitness = float('-inf')
        self.fitness_history = []
        self.convergence_history = []

    def fitness_function(self, position, X, y, threshold=0.50):
        selected_features = position > threshold
        n_selected = np.sum(selected_features)

        if n_selected < 100:  # Minimum features
            return -1

        try:
            X_selected = X[:, selected_features]

            X_train, X_test, y_train, y_test = train_test_split(
                X_selected, y, test_size=0.20, random_state=self.random_state,
                stratify=y
            )

            # Use both ETC and XGBoost for fitness
            etc = ExtraTreesClassifier(
                n_estimators=150,
                max_depth=20,
                random_state=self.random_state,
                n_jobs=-1,
                class_weight='balanced'
            )
            etc.fit(X_train, y_train)
            y_pred_etc = etc.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred_etc)
            f1 = f1_score(y_test, y_pred_etc)
            precision = precision_score(y_test, y_pred_etc)
            recall = recall_score(y_test, y_pred_etc)

            feature_ratio = n_selected / len(position)
            reduction_bonus = (1 - feature_ratio) * 0.1

            # Balanced fitness emphasizing accuracy and F1
            fitness = 0.45 * accuracy + 0.35 * f1 + 0.10 * precision + 0.05 * recall + reduction_bonus

            return fitness

        except Exception as e:
            return -1

    def update_position(self, hawk_pos, best_pos, iteration, max_iter):
        E0 = 2 * np.random.random() - 1
        E = 2 * E0 * (1 - iteration / max_iter)

        if abs(E) >= 1:
            # Exploration phase
            if np.random.random() < 0.5:
                rand_hawk_idx = np.random.randint(0, self.n_hawks)
                r1, r2 = np.random.random(2)
                new_pos = np.random.random(len(hawk_pos)) - r1 * abs(np.random.random(len(hawk_pos)) - 2 * r2 * hawk_pos)
            else:
                r3, r4 = np.random.random(2)
                new_pos = (best_pos - np.mean(hawk_pos)) - r3 * (2 * r4 * hawk_pos)
        else:
            # Exploitation phase
            r = np.random.random()
            if r >= 0.5 and abs(E) >= 0.5:
                delta_X = best_pos - hawk_pos
                new_pos = delta_X - E * abs(np.random.random() * best_pos - hawk_pos)
            elif r >= 0.5 and abs(E) < 0.5:
                new_pos = best_pos - E * abs(best_pos - hawk_pos)
            elif r < 0.5 and abs(E) >= 0.5:
                delta_X = best_pos - hawk_pos
                S = np.random.random(len(hawk_pos)) * 2 * np.pi
                new_pos = delta_X - E * abs(np.cos(S) * best_pos - hawk_pos)
            else:
                S = np.random.random(len(hawk_pos)) * 2 * np.pi
                new_pos = best_pos - E * abs(np.cos(S) * best_pos - hawk_pos)

        new_pos = np.clip(new_pos, 0, 1)
        return new_pos

    def optimize(self, X, y):
        print(f"Starting Harris Hawks Optimization...")
        print(f"Hawks: {self.n_hawks}, Iterations: {self.max_iterations}")

        # Sample for faster optimization
        if X.shape[0] > 2000:
            sample_indices = np.random.choice(X.shape[0], 2000, replace=False)
            X_sample = X[sample_indices]
            y_sample = y[sample_indices]
        else:
            X_sample, y_sample = X, y

        n_features = X.shape[1]
        hawks = np.random.uniform(0, 1, (self.n_hawks, n_features))
        fitness_values = np.zeros(self.n_hawks)

        for iteration in range(self.max_iterations):
            for i in range(self.n_hawks):
                fitness_values[i] = self.fitness_function(hawks[i], X_sample, y_sample)

            best_idx = np.argmax(fitness_values)
            current_best_fitness = fitness_values[best_idx]

            if current_best_fitness > self.best_fitness:
                self.best_fitness = current_best_fitness
                self.best_position = hawks[best_idx].copy()

            self.fitness_history.append(self.best_fitness)
            self.convergence_history.append(np.mean(fitness_values))

            for i in range(self.n_hawks):
                hawks[i] = self.update_position(hawks[i], self.best_position, iteration, self.max_iterations)

            if (iteration + 1) % 5 == 0:
                print(f"Iteration {iteration + 1}: Best fitness = {self.best_fitness:.4f}")

        return self.best_position

class EnhancedFakeReviewDetectionSystem:
    def __init__(self, config=Config()):
        self.config = config
        self.preprocessor = AdvancedTextPreprocessor()
        self.feature_extractor = EnhancedFeatureExtractor(config)
        self.optimizer = HarrisHawksOptimizer(
            n_hawks=config.N_HAWKS,
            max_iterations=config.MAX_ITERATIONS,
            random_state=config.RANDOM_STATE
        )
        self.model = None
        self.selected_features = None
        self.results = {}

    def load_and_preprocess_data(self, df):
        print("\n" + "="*60)
        print("DATA PREPROCESSING")
        print("="*60)

        print(f"Dataset shape: {df.shape}")
        print(f"\nLabel distribution:")
        print(df['label'].value_counts())

        print("\nApplying advanced preprocessing...")
        df['processed_text'] = df['text_'].apply(self.preprocessor.preprocess)

        df = df[df['processed_text'].str.len() > 0].reset_index(drop=True)
        print(f"After cleaning: {df.shape}")

        return df

    def extract_features_and_optimize(self, df):
        print("\n" + "="*60)
        print("FEATURE EXTRACTION & OPTIMIZATION")
        print("="*60)

        X = self.feature_extractor.fit_transform(df['processed_text'].tolist())
        y = df['label'].map({'OR': 0, 'CG': 1}).values

        print(f"Initial features: {X.shape}")
        print(f"Class distribution: {Counter(y)}")

        # Use SMOTEENN for better handling
        print("\nApplying SMOTEENN for balanced dataset...")

        # Calculate k_neighbors safely
        min_class_count = Counter(y).most_common()[-1][1]
        k_neighbors = min(self.config.SMOTE_K_NEIGHBORS, min_class_count - 1)
        if k_neighbors < 1:
            print(f"Warning: k_neighbors is < 1 ({k_neighbors}). Setting to 1.")
            k_neighbors = 1

        if self.config.USE_SMOTE_ENN:
            smote_enn = SMOTEENN(
                random_state=self.config.RANDOM_STATE,
                smote=SMOTE(  # <-- CORRECTED: Must be SMOTE, not BorderlineSMOTE
                    k_neighbors=k_neighbors,
                    random_state=self.config.RANDOM_STATE
                )
            )
            X_balanced, y_balanced = smote_enn.fit_resample(X, y)
        else:
            smote = SMOTE(
                random_state=self.config.RANDOM_STATE,
                k_neighbors=k_neighbors
            )
            X_balanced, y_balanced = smote.fit_resample(X, y)

        print(f"After balancing: {Counter(y_balanced)}")

        print("\nApplying Harris Hawks Optimization...")
        X_dense = X_balanced.toarray() if hasattr(X_balanced, 'toarray') else X_balanced

        best_features = self.optimizer.optimize(X_dense, y_balanced)
        self.selected_features = best_features > self.config.FEATURE_THRESHOLD

        n_selected = np.sum(self.selected_features)
        print(f"Selected {n_selected}/{len(best_features)} features ({n_selected/len(best_features)*100:.1f}%)")

        X_selected = X_balanced[:, self.selected_features]

        return X_selected, y_balanced

    def train_ensemble_model(self, X, y):
        print("\n" + "="*60)
        print("ENSEMBLE MODEL TRAINING")
        print("="*60)

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self.config.TEST_SIZE,
            random_state=self.config.RANDOM_STATE, stratify=y
        )

        print(f"Training: {X_train.shape}, Testing: {X_test.shape}")

        # Convert to dense
        if hasattr(X_train, 'toarray'):
            print("Converting sparse matrices to dense format...")
            X_train_dense = X_train.toarray()
            X_test_dense = X_test.toarray()
        else:
            X_train_dense = X_train
            X_test_dense = X_test

        # Optimized Extra Trees Classifier
        etc = ExtraTreesClassifier(
            n_estimators=self.config.ETC_N_ESTIMATORS,
            max_depth=self.config.ETC_MAX_DEPTH,
            min_samples_split=self.config.ETC_MIN_SAMPLES_SPLIT,
            min_samples_leaf=self.config.ETC_MIN_SAMPLES_LEAF,
            max_features=self.config.ETC_MAX_FEATURES,
            n_jobs=-1,
            random_state=self.config.RANDOM_STATE,
            class_weight='balanced',
            bootstrap=True,
            oob_score=False
        )

        # Optimized Random Forest
        rf = RandomForestClassifier(
            n_estimators=400,
            max_depth=30,
            min_samples_split=3,
            min_samples_leaf=1,
            max_features='sqrt',
            n_jobs=-1,
            random_state=self.config.RANDOM_STATE,
            class_weight='balanced'
        )

        # Optimized XGBoost
        if XGBOOST_AVAILABLE:
            xgb_clf = xgb.XGBClassifier(
                n_estimators=self.config.XGB_N_ESTIMATORS,
                max_depth=self.config.XGB_MAX_DEPTH,
                learning_rate=self.config.XGB_LEARNING_RATE,
                subsample=self.config.XGB_SUBSAMPLE,
                colsample_bytree=self.config.XGB_COLSAMPLE_BYTREE,
                min_child_weight=self.config.XGB_MIN_CHILD_WEIGHT,
                gamma=self.config.XGB_GAMMA,
                random_state=self.config.RANDOM_STATE,
                use_label_encoder=False,
                eval_metric='logloss',
                scale_pos_weight=1,
                n_jobs=4
            )

        # Optimized SVM
        base_svm = LinearSVC(
            C=self.config.SVM_C,
            random_state=self.config.RANDOM_STATE,
            max_iter=3000,
            class_weight='balanced',
            dual='auto'
        )
        svm_clf = CalibratedClassifierCV(base_svm, cv=3)

        print("Training Advanced Ensemble Model...")
        start_time = time.time()

        if XGBOOST_AVAILABLE:
            # Advanced Stacking with multiple base learners
            estimators = [
                ('etc', etc),
                ('rf', rf),
                ('xgb', xgb_clf),
                ('svm', svm_clf)
            ]

            # Use Logistic Regression with optimized parameters as meta-learner
            final_estimator = LogisticRegression(
                C=1.0,
                max_iter=2000,
                random_state=self.config.RANDOM_STATE,
                class_weight='balanced'
            )

            self.model = StackingClassifier(
                estimators=estimators,
                final_estimator=final_estimator,
                cv=5,
                n_jobs=2,
                passthrough=False
            )
        else:
            # Voting classifier fallback
            estimators = [
                ('etc', etc),
                ('rf', rf),
                ('svm', svm_clf)
            ]

            self.model = VotingClassifier(
                estimators=estimators,
                voting='soft',
                n_jobs=2,
                weights=[1.2, 1.0, 0.8]  # Weight towards ETC
            )

        self.model.fit(X_train_dense, y_train)
        training_time = time.time() - start_time

        start_time = time.time()
        y_pred = self.model.predict(X_test_dense)
        y_proba = self.model.predict_proba(X_test_dense)
        prediction_time = time.time() - start_time

        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Stratified K-Fold cross-validation
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.config.RANDOM_STATE)
        cv_scores = cross_val_score(self.model, X_train_dense, y_train, cv=skf, scoring='accuracy')
        cv_f1_scores = cross_val_score(self.model, X_train_dense, y_train, cv=skf, scoring='f1')

        self.results = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'cv_accuracy_mean': cv_scores.mean(),
            'cv_accuracy_std': cv_scores.std(),
            'cv_f1_mean': cv_f1_scores.mean(),
            'cv_f1_std': cv_f1_scores.std(),
            'training_time': training_time,
            'prediction_time': prediction_time
        }

        print(f"\n{'='*60}")
        print("RESULTS:")
        print(f"{'='*60}")
        print(f"Accuracy:  {accuracy:.6f} ({accuracy*100:.2f}%)")
        print(f"Precision: {precision:.6f} ({precision*100:.2f}%)")
        print(f"Recall:    {recall:.6f} ({recall*100:.2f}%)")
        print(f"F1-Score:  {f1:.6f} ({f1*100:.2f}%)")
        print(f"CV Accuracy: {cv_scores.mean():.6f} ± {cv_scores.std():.4f}")
        print(f"CV F1: {cv_f1_scores.mean():.6f} ± {cv_f1_scores.std():.4f}")
        print(f"Training Time: {training_time:.2f}s")
        print(f"{'='*60}")

        print(f"\nClassification Report:")
        print(classification_report(y_test, y_pred, target_names=['Real (OR)', 'Fake (CG)']))

        return X_test, y_test, y_pred, y_proba

    def predict(self, text):
        processed_text = self.preprocessor.preprocess(text)
        features = self.feature_extractor.transform([processed_text])
        features_selected = features[:, self.selected_features]

        if hasattr(features_selected, 'toarray'):
            features_selected = features_selected.toarray()

        prediction = self.model.predict(features_selected)[0]
        probabilities = self.model.predict_proba(features_selected)[0]

        return {
            'original_text': text,
            'processed_text': processed_text,
            'prediction': 'Fake (CG)' if prediction == 1 else 'Real (OR)',
            'confidence': max(probabilities),
            'probabilities': {
                'Real (OR)': probabilities[0],
                'Fake (CG)': probabilities[1]
            }
        }

    def save_model(self, filepath='optimized_fake_review_model.pkl'):
        model_data = {
            'preprocessor': self.preprocessor,
            'feature_extractor': self.feature_extractor,
            'selected_features': self.selected_features,
            'model': self.model,
            'config': self.config,
            'results': self.results
        }

        with open(filepath, 'wb') as f:
            pickle.dump(model_data, f)
        print(f"✓ Model saved to {filepath}")

    def load_model(self, filepath='optimized_fake_review_model.pkl'):
        with open(filepath, 'rb') as f:
            model_data = pickle.load(f)

        self.preprocessor = model_data['preprocessor']
        self.feature_extractor = model_data['feature_extractor']
        self.selected_features = model_data['selected_features']
        self.model = model_data['model']
        self.config = model_data['config']
        self.results = model_data.get('results', {})

        print(f"✓ Model loaded from {filepath}")

def create_enhanced_visualizations(detection_system, y_test, y_pred, y_proba):
    """Create comprehensive visualization suite"""

    # Figure 1: Performance Metrics Comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 11))

    # Metrics bar chart
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
    values = [
        detection_system.results['accuracy'],
        detection_system.results['precision'],
        detection_system.results['recall'],
        detection_system.results['f1']
    ]
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    bars = axes[0, 0].bar(metrics, values, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
    axes[0, 0].set_title('Performance Metrics', fontsize=13, fontweight='bold')
    axes[0, 0].set_ylabel('Score', fontsize=11, fontweight='bold')
    axes[0, 0].set_ylim(0, 1.1)
    axes[0, 0].grid(axis='y', alpha=0.3, linestyle='--')
    axes[0, 0].axhline(y=0.95, color='red', linestyle='--', linewidth=2.5, label='95% Target', alpha=0.8)
    axes[0, 0].legend(fontsize=10)

    for bar, value in zip(bars, values):
        height = bar.get_height()
        axes[0, 0].text(bar.get_x() + bar.get_width()/2, height + 0.02,
                  f'{value:.4f}\n({value*100:.2f}%)', ha='center', va='bottom',
                  fontweight='bold', fontsize=10)

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
                xticklabels=['Real (OR)', 'Fake (CG)'],
                yticklabels=['Real (OR)', 'Fake (CG)'],
                square=True, linewidths=2, linecolor='black', ax=axes[0, 1],
                annot_kws={'fontsize': 14, 'fontweight': 'bold'})
    axes[0, 1].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
    axes[0, 1].set_ylabel('True Label', fontsize=11, fontweight='bold')
    axes[0, 1].set_xlabel('Predicted Label', fontsize=11, fontweight='bold')

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_proba[:, 1])
    roc_auc = auc(fpr, tpr)
    axes[1, 0].plot(fpr, tpr, color='darkorange', lw=3, label=f'ROC (AUC = {roc_auc:.4f})')
    axes[1, 0].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
    axes[1, 0].fill_between(fpr, tpr, alpha=0.2, color='orange')
    axes[1, 0].set_xlim([0.0, 1.0])
    axes[1, 0].set_ylim([0.0, 1.05])
    axes[1, 0].set_xlabel('False Positive Rate', fontsize=11, fontweight='bold')
    axes[1, 0].set_ylabel('True Positive Rate', fontsize=11, fontweight='bold')
    axes[1, 0].set_title(f'ROC Curve (AUC = {roc_auc:.4f})', fontsize=13, fontweight='bold')
    axes[1, 0].legend(loc="lower right", fontsize=10)
    axes[1, 0].grid(True, alpha=0.3, linestyle='--')

    # HHO Convergence
    axes[1, 1].plot(detection_system.optimizer.fitness_history, 'b-', linewidth=2.5,
                    label='Best Fitness', marker='o', markersize=3, markevery=5)
    axes[1, 1].plot(detection_system.optimizer.convergence_history, 'r--', linewidth=2,
                    alpha=0.7, label='Avg Fitness')
    axes[1, 1].fill_between(range(len(detection_system.optimizer.fitness_history)),
                            detection_system.optimizer.fitness_history,
                            alpha=0.2, color='blue')
    axes[1, 1].set_title('HHO Convergence', fontsize=13, fontweight='bold')
    axes[1, 1].set_xlabel('Iteration', fontsize=11, fontweight='bold')
    axes[1, 1].set_ylabel('Fitness Score', fontsize=11, fontweight='bold')
    axes[1, 1].legend(fontsize=10)
    axes[1, 1].grid(True, alpha=0.3, linestyle='--')

    plt.suptitle('Optimized Fake Review Detection System - Performance Analysis',
                 fontsize=15, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/optimized_performance_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: optimized_performance_analysis.png")

    # Figure 2: Feature Analysis
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    # Feature selection pie chart
    total_features = len(detection_system.selected_features)
    selected_count = np.sum(detection_system.selected_features)
    rejected_count = total_features - selected_count

    colors = ['#4CAF50', '#F44336']
    explode = (0.05, 0)
    wedges, texts, autotexts = axes[0].pie([selected_count, rejected_count],
            labels=[f'Selected\n{selected_count}\n({selected_count/total_features*100:.1f}%)',
                    f'Rejected\n{rejected_count}\n({rejected_count/total_features*100:.1f}%)'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            explode=explode, shadow=True, textprops={'fontweight': 'bold', 'fontsize': 11})
    axes[0].set_title('Feature Selection Summary by HHO', fontsize=13, fontweight='bold')

    # Confidence distribution
    confidence_scores = np.max(y_proba, axis=1)
    axes[1].hist(confidence_scores, bins=40, alpha=0.7, color='purple', edgecolor='black', linewidth=1.5)
    axes[1].set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Confidence Score', fontsize=11, fontweight='bold')
    axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
    axes[1].grid(True, alpha=0.3, linestyle='--')
    axes[1].axvline(np.mean(confidence_scores), color='red', linestyle='--',
                    linewidth=2.5, label=f'Mean: {np.mean(confidence_scores):.3f}', alpha=0.8)
    axes[1].axvline(np.median(confidence_scores), color='green', linestyle='--',
                    linewidth=2.5, label=f'Median: {np.median(confidence_scores):.3f}', alpha=0.8)
    axes[1].legend(fontsize=10)

    plt.suptitle('Feature Analysis and Prediction Confidence', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/optimized_feature_analysis.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: optimized_feature_analysis.png")

def create_results_table(detection_system):
    """Create detailed results table"""
    fig, ax = plt.subplots(figsize=(14, 7))
    ax.axis('tight')
    ax.axis('off')

    table_data = [
        ['Accuracy', f"{detection_system.results['accuracy']:.6f}",
         f"{detection_system.results['accuracy']*100:.2f}%",
         '✓ PASSED' if detection_system.results['accuracy'] >= 0.95 else '✗ BELOW TARGET'],
        ['Precision', f"{detection_system.results['precision']:.6f}",
         f"{detection_system.results['precision']*100:.2f}%",
         '✓ PASSED' if detection_system.results['precision'] >= 0.95 else '✗ BELOW TARGET'],
        ['Recall', f"{detection_system.results['recall']:.6f}",
         f"{detection_system.results['recall']*100:.2f}%",
         '✓ PASSED' if detection_system.results['recall'] >= 0.95 else '✗ BELOW TARGET'],
        ['F1-Score', f"{detection_system.results['f1']:.6f}",
         f"{detection_system.results['f1']*100:.2f}%",
         '✓ PASSED' if detection_system.results['f1'] >= 0.95 else '✗ BELOW TARGET'],
        ['CV Accuracy', f"{detection_system.results['cv_accuracy_mean']:.6f} ± {detection_system.results['cv_accuracy_std']:.4f}",
         f"{detection_system.results['cv_accuracy_mean']*100:.2f}%",
         '✓ PASSED' if detection_system.results['cv_accuracy_mean'] >= 0.95 else '✗ BELOW TARGET'],
        ['CV F1-Score', f"{detection_system.results['cv_f1_mean']:.6f} ± {detection_system.results['cv_f1_std']:.4f}",
         f"{detection_system.results['cv_f1_mean']*100:.2f}%", '-'],
        ['Training Time', f"{detection_system.results['training_time']:.2f} seconds", '-', '-'],
        ['Prediction Time', f"{detection_system.results['prediction_time']:.4f} seconds", '-', '-']
    ]

    table = ax.table(cellText=table_data,
                     colLabels=['Metric', 'Value', 'Percentage', 'Status (Target: >95%)'],
                     cellLoc='center',
                     loc='center',
                     colWidths=[0.25, 0.30, 0.20, 0.25])

    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2.8)

    for i in range(len(table_data) + 1):
        for j in range(4):
            cell = table[(i, j)]
            if i == 0:
                cell.set_facecolor('#2196F3')
                cell.set_text_props(weight='bold', color='white', fontsize=12)
            else:
                if j == 3 and i <= 5:
                    if '✓ PASSED' in table_data[i-1][3]:
                        cell.set_facecolor('#90EE90')
                        cell.set_text_props(weight='bold')
                    elif '✗ BELOW' in table_data[i-1][3]:
                        cell.set_facecolor('#FFB6C6')
                        cell.set_text_props(weight='bold')
                    else:
                        cell.set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')
                else:
                    cell.set_facecolor('#f0f0f0' if i % 2 == 0 else 'white')

    plt.title('Optimized Model Performance Metrics (Target: >95% Accuracy)',
              fontsize=15, fontweight='bold', pad=25)
    plt.savefig(f'{OUTPUT_DIR}/optimized_results_table.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✓ Saved: optimized_results_table.png")

def main():
    print("="*70)
    print("OPTIMIZED FAKE REVIEW DETECTION SYSTEM")
    print("HHO + Advanced Ensemble Learning (ETC + RF + XGBoost + SVM)")
    print("Target: >95% Accuracy")
    print("="*70)

    config = Config()
    detection_system = EnhancedFakeReviewDetectionSystem(config)

    df = load_dataset()
    if df is None:
        print("Failed to load dataset. Exiting...")
        return None

    df = detection_system.load_and_preprocess_data(df)

    X_selected, y_balanced = detection_system.extract_features_and_optimize(df)

    start_time = time.time()
    X_test, y_test, y_pred, y_proba = detection_system.train_ensemble_model(X_selected, y_balanced)
    total_training_time = time.time() - start_time

    create_enhanced_visualizations(detection_system, y_test, y_pred, y_proba)
    create_results_table(detection_system)

    print("\n" + "="*60)
    print("TESTING ON SAMPLE REVIEWS")
    print("="*60)

    test_reviews = [
        "I've been using this product for about 3 months and it works well. Good build quality.",
        "Amazing! Perfect! Love love love! Best ever! Highly recommend to everyone!!!",
        "Decent value for the price. Works as expected, no major issues so far.",
        "Excellent excellent excellent! Outstanding quality! Five stars! Super recommend!",
        "The product arrived on time and packaging was good. Performance is as described.",
        "WOW!!! AMAZING!!! BEST PURCHASE EVER!!! 100% RECOMMENDED!!! FIVE STARS!!!",
        "This product has some pros and cons. Works okay but could be better.",
        "PERFECT PERFECT PERFECT!!! BUY IT NOW!!! EVERYONE NEEDS THIS!!!"
    ]

    for i, review in enumerate(test_reviews, 1):
        try:
            result = detection_system.predict(review)
            print(f"\nTest {i}:")
            print(f"Text: {result['original_text'][:70]}...")
            print(f"Prediction: {result['prediction']}")
            print(f"Confidence: {result['confidence']:.4f}")
            print(f"Probabilities: Real={result['probabilities']['Real (OR)']:.4f}, "
                  f"Fake={result['probabilities']['Fake (CG)']:.4f}")
        except Exception as e:
            print(f"Error: {e}")

    model_path = f'{OUTPUT_DIR}/optimized_fake_review_model.pkl'
    detection_system.save_model(model_path)

    print("\n" + "="*70)
    print("FINAL SUMMARY")
    print("="*70)
    print(f"✓ Dataset: {df.shape[0]} reviews processed")
    print(f"✓ Features: {len(detection_system.selected_features)} → "
          f"{np.sum(detection_system.selected_features)} selected by HHO")
    print(f"✓ Accuracy: {detection_system.results['accuracy']*100:.2f}% "
          f"{'✓ TARGET ACHIEVED!' if detection_system.results['accuracy'] >= 0.95 else '✗ Below 95%'}")
    print(f"✓ Precision: {detection_system.results['precision']*100:.2f}%")
    print(f"✓ Recall: {detection_system.results['recall']*100:.2f}%")
    print(f"✓ F1-Score: {detection_system.results['f1']*100:.2f}% "
          f"{'✓ TARGET ACHIEVED!' if detection_system.results['f1'] >= 0.95 else '✗ Below 95%'}")
    print(f"✓ CV Accuracy: {detection_system.results['cv_accuracy_mean']*100:.2f}%")
    print(f"✓ Model saved: {model_path}")
    print(f"✓ All visualizations saved in '{OUTPUT_DIR}' folder")

    if detection_system.results['accuracy'] >= 0.95:
        print("\n" + "🎉"*30)
        print("SUCCESS! Model achieved >95% accuracy target!")
        print("🎉"*30)
    else:
        print("\n⚠ Model performance analysis:")
        print(f"  - Current accuracy: {detection_system.results['accuracy']*100:.2f}%")
        print(f"  - Gap to target: {(0.95 - detection_system.results['accuracy'])*100:.2f}%")
        print("\n  Suggestions for improvement:")
        print("  1. Increase N_HAWKS to 40-50 for better feature selection")
        print("  2. Increase MAX_ITERATIONS to 60-80")
        print("  3. Try different SMOTE variants (ADASYN, BorderlineSMOTE)")
        print("  4. Adjust ensemble weights in VotingClassifier")
        print("  5. Perform hyperparameter grid search")

    print("\n" + "="*70)
    print("PROCESS COMPLETED!")
    print("="*70)

    return detection_system

if __name__ == "__main__":
    detection_system = main()

Libraries loaded successfully!
OPTIMIZED FAKE REVIEW DETECTION SYSTEM
HHO + Advanced Ensemble Learning (ETC + RF + XGBoost + SVM)
Target: >95% Accuracy
Dataset loaded from reviews.csv
Shape: (40432, 4)

DATA PREPROCESSING
Dataset shape: (40432, 4)

Label distribution:
label
CG    20216
OR    20216
Name: count, dtype: int64

Applying advanced preprocessing...
After cleaning: (40431, 5)

FEATURE EXTRACTION & OPTIMIZATION
Extracting TF-IDF features...
Extracting character n-gram features...
Extracting count features...
Extracting advanced linguistic features...
Combining all features...
Initial features: (40431, 13539)
Class distribution: Counter({np.int64(0): 20216, np.int64(1): 20215})

Applying SMOTEENN for balanced dataset...
After balancing: Counter({np.int64(0): 13753, np.int64(1): 8152})

Applying Harris Hawks Optimization...
Starting Harris Hawks Optimization...
Hawks: 30, Iterations: 50
Iteration 5: Best fitness = 0.9416
Iteration 10: Best fitness = 0.9455
Iteration 15: Best fitn